# DCPA-Flat v3.3 — Colab 省积分全链路控制台

支持 **CPU 零 GPU 积分多核造数 (`cpu_forge`) → 48M 5,000 步三路生死门 (`killgate_5k`) → 48M / 206M 五阶段旗舰训练 (`train_48m` / `train_206m`) → Rust 零拷贝导出与 Gate 6 验收 (`export_and_gate6`)**。

- 架构规范：`docs/DCPA_FLAT_V3_3_SPEC.md`
- 分支：`feat/dcpa-flat-v3-3`
- 推荐执行顺序：先租 **无 GPU 高内存 CPU 实例** 跑 Cell 1~3 造数打包，再租 **RTX Pro 6000 (96GB) / A100** 跑 `killgate_5k`，绿灯后再启动 `train_206m`。

In [ ]:
# Cell 1 [全局控制台：运行模式 / 语料档位 / 归档存储 / 挂载 Drive & CPU/GPU 体检]
# ==============================================================================
# RUN_MODE 可选值：
#   "cpu_forge"        -> 【阶段 0：零 GPU 积分】租无 GPU 高内存 CPU 机，多线程拉取 2025-2026 精选语料 +
#                         执行无损分块 + 合成推理/写作/代码/工具/SFT 炼金分片并打包 .tar.zst 归档
#   "killgate_5k"      -> 【阶段 1：~25 分钟 GPU】48M 5,000 步三路同源对比 (Run A vs B vs C) + Gate A~E 体检
#   "train_48m"        -> 【阶段 2A：端侧量产】DCPA-Flat-48M 五阶段完整训练 (configs/v3/dcpa_flat_48m_v3.yaml)
#   "train_206m"       -> 【阶段 2B：旗舰底座】DCPA-Flat-206M 五阶段旗舰训练 (configs/v3/dcpa_flat_206m_v3.yaml)
#   "export_and_gate6" -> 【阶段 3：双端闭环】Rust 零拷贝导出验证 + 领域插件零漂移微调 + Gate 6 验收
#
# DATA_PROFILE 可选值（由 prepare_vast_corpus.py 自动构建并校验）：
#   "v3_flat_5g"   -> ~5.5 GB（最快验证 5k 生死门，CPU 约 12~20 分钟；压缩后 ~2.3 GB，可存 Drive）
#   "v3_flat_18g"  -> ~18 GB（48M 端侧量产 / 32GB 流，压缩后 ~7.5 GB）
#   "v3_flat_45g"  -> ~45 GB（206M Takeoff 80GB 流，压缩后 ~19 GB）
#   "v3_flat_90g"  -> ~90 GB（一步到位覆盖 48M + 206M 完整 160GB 旗舰流，压缩后 ~38 GB，推荐存 HF 私有库）
#   "v3_flat_160g" -> ~160 GB（一步到位同时覆盖未来 650M，压缩后 ~68 GB，强烈建议存 HF 私有库 + 250GB+ 磁盘）
# ==============================================================================
RUN_MODE = "cpu_forge"               # 先选 "cpu_forge"，造数完切 GPU 后改 "killgate_5k" -> "train_206m"
DATA_PROFILE = "v3_flat_5g"          # 一步到位可改 "v3_flat_90g"（注意 Drive 剩余空间或切 ARCHIVE_STORAGE="hf_repo"）
ARCHIVE_STORAGE = "drive"            # "drive" (Google Drive) 或 "hf_repo" (免费 HuggingFace 私有 Dataset，极速且不占 Drive)
HF_ARCHIVE_REPO = ""                 # 当 ARCHIVE_STORAGE="hf_repo" 时填写，如 "Eveyz/dcpa-flat-v3-corpus"
DRIVE_ROOT_STR = "/content/drive/MyDrive/DCPA_Flat_v3"

# 代码仓库与实验追踪
GIT_REPO_URL = "https://github.com/Eveyz/dcpa.git"
GIT_BRANCH = "feat/dcpa-flat-v3-3"
USE_WANDB = True
WANDB_PROJECT = "dcpa-flat-v3"
WANDB_RUN_NAME = ""                  # 留空则自动按模式命名；Colab 断连重连自动续写同一 run_id

# 架构裁决覆写（跑完 killgate_5k 后按 Gate B / Gate D 结果锁定，留空则沿用 YAML 默认值）
USE_LOCAL_WINDOW_LAYERS = ""         # "" | "true" (L1+L4+L6 W=128) | "false" (纯 K=8)
CARRIER_GRAD_GUARD = ""              # "" | "none" | "router_sg" | "soft_clip"
TOTAL_STEPS_OVERRIDE = ""            # 留空按 YAML（206M Takeoff=305000；后续无痛扩训 160GB 旗舰可改为 "610000"）
EXTRA_TRAIN_ARGS = ""                # 可选透传 train_flat.py 任意额外参数，如 "--grad-accum-steps 2"

# 省积分保险丝：任务成功完工且 os.sync() 落盘后自动归还 Colab 实例（CPU/GPU 均生效）
AUTO_UNASSIGN_ON_FINISH = True
MOUNT_DRIVE = True                   # 若仅用 hf_repo 也可保持 True，用于保存 A/B 断点与日志
HF_TOKEN = ""                        # 留空自动读 Colab Secrets (🔑) 中的 HF_TOKEN

import os
import shutil
import subprocess
from pathlib import Path

VALID_MODES = ("cpu_forge", "killgate_5k", "train_48m", "train_206m", "export_and_gate6")
VALID_PROFILES = ("v3_flat_5g", "v3_flat_18g", "v3_flat_45g", "v3_flat_90g", "v3_flat_160g")
if RUN_MODE not in VALID_MODES:
    raise ValueError(f"RUN_MODE must be one of {VALID_MODES}, got {RUN_MODE!r}")
if DATA_PROFILE not in VALID_PROFILES:
    raise ValueError(f"DATA_PROFILE must be one of {VALID_PROFILES}, got {DATA_PROFILE!r}")
if ARCHIVE_STORAGE not in ("drive", "hf_repo"):
    raise ValueError("ARCHIVE_STORAGE must be 'drive' or 'hf_repo'")
if ARCHIVE_STORAGE == "hf_repo" and not HF_ARCHIVE_REPO.strip():
    raise ValueError("ARCHIVE_STORAGE='hf_repo' requires HF_ARCHIVE_REPO (e.g. 'username/dcpa-flat-v3-corpus')")

os.environ["DCPA_FLAT_RUN_MODE"] = RUN_MODE
os.environ["DCPA_FLAT_DATA_PROFILE"] = DATA_PROFILE
os.environ["DCPA_FLAT_ARCHIVE_STORAGE"] = ARCHIVE_STORAGE
os.environ["DCPA_FLAT_HF_ARCHIVE_REPO"] = HF_ARCHIVE_REPO.strip()
os.environ["DCPA_FLAT_DRIVE_ROOT"] = DRIVE_ROOT_STR
os.environ["DCPA_FLAT_GIT_URL"] = GIT_REPO_URL.strip()
os.environ["DCPA_FLAT_GIT_BRANCH"] = GIT_BRANCH.strip()
os.environ["DCPA_FLAT_USE_WANDB"] = "1" if USE_WANDB else "0"
os.environ["WANDB_PROJECT"] = WANDB_PROJECT
if WANDB_RUN_NAME.strip():
    os.environ["WANDB_NAME"] = WANDB_RUN_NAME.strip()
os.environ["DCPA_FLAT_LOCAL_WINDOW"] = USE_LOCAL_WINDOW_LAYERS.strip().lower()
os.environ["DCPA_FLAT_GRAD_GUARD"] = CARRIER_GRAD_GUARD.strip().lower()
os.environ["DCPA_FLAT_TOTAL_STEPS"] = TOTAL_STEPS_OVERRIDE.strip()
os.environ["DCPA_FLAT_EXTRA_ARGS"] = EXTRA_TRAIN_ARGS.strip()
os.environ["DCPA_FLAT_AUTO_UNASSIGN"] = "1" if AUTO_UNASSIGN_ON_FINISH else "0"
if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN.strip()

if MOUNT_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    drive_root = Path(DRIVE_ROOT_STR)
    for sub in (
        "data_archive",
        "checkpoints/flat_48m",
        "checkpoints/flat_206m",
        "logs/killgate_5k",
        "logs/flat_48m",
        "logs/flat_206m",
        "export/flat_48m",
        "export/flat_206m",
    ):
        (drive_root / sub).mkdir(parents=True, exist_ok=True)
    print(f"[Drive] Ready: {drive_root}")

# CPU / RAM / 本地 NVMe 磁盘与 GPU 体检（兼容无 GPU 的纯 CPU 实例）
cpu_cores = os.cpu_count() or 2
disk = shutil.disk_usage("/content")
free_gb = disk.free / (1024**3)
ram_gb = 0.0
try:
    with open("/proc/meminfo", "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("MemTotal:"):
                ram_gb = float(line.split()[1]) / (1024**2)
                break
except Exception:
    pass

print(f"[Host] RUN_MODE={RUN_MODE} | DATA_PROFILE={DATA_PROFILE} | STORAGE={ARCHIVE_STORAGE}")
print(f"[Host] CPU={cpu_cores} cores | RAM={ram_gb:.1f} GB | Local /content free={free_gb:.1f} GB")

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True,
    text=True,
    check=False,
)
if smi.returncode == 0 and smi.stdout.strip():
    gpu_name, mem_mib_s = [p.strip() for p in smi.stdout.strip().splitlines()[0].split(",", 1)]
    vram_gb = float(mem_mib_s) / 1024.0
    if RUN_MODE == "cpu_forge":
        print(f"[WARN] 当前检测到 GPU ({gpu_name}, {vram_gb:.1f} GB)，但 RUN_MODE='cpu_forge'！建议切回无 GPU 实例以节省积分。")
    else:
        gc_hint = "OFF (96G/80G 全速直跑)" if vram_gb >= 45.0 else "ON (自动开启梯度检查点)"
        print(f"[GPU] {gpu_name} | VRAM={vram_gb:.1f} GB | grad_checkpoint={gc_hint} | bf16=ON")
else:
    if RUN_MODE in ("killgate_5k", "train_48m", "train_206m"):
        print("[WARN] 未检测到 GPU！运行 GPU 训练前请在 Colab 菜单切换为 RTX Pro 6000 / A100 实例。")
    else:
        print("[CPU Mode] 纯 CPU 实例就绪（零 GPU 积分消耗），适合多核语料抓取、无损分块与合成炼金。")


In [ ]:
# Cell 2 [拉取 feat/dcpa-flat-v3-3 代码至本地 NVMe + 安装依赖与 zstd 多线程工具]
import base64
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

CODE_DIR = Path("/content/dcpa")
NVME_ROOT = Path("/content/nvme_cache")
NVME_ROOT.mkdir(parents=True, exist_ok=True)

repo_url = os.environ.get("DCPA_FLAT_GIT_URL", "https://github.com/Eveyz/dcpa.git")
git_branch = os.environ.get("DCPA_FLAT_GIT_BRANCH", "feat/dcpa-flat-v3-3")
run_mode = os.environ.get("DCPA_FLAT_RUN_MODE", "cpu_forge")

# 读取 Colab Secrets (🔑)
def _load_secret(name: str) -> str:
    val = os.environ.get(name, "").strip()
    if val:
        return val
    try:
        from google.colab import userdata

        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

gh_token = _load_secret("GITHUB_TOKEN")
wb_key = _load_secret("WANDB_API_KEY")
if wb_key:
    os.environ["WANDB_API_KEY"] = wb_key
hf_tok = _load_secret("HF_TOKEN") or _load_secret("HUGGING_FACE_HUB_TOKEN")
if hf_tok:
    os.environ["HF_TOKEN"] = hf_tok
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_tok

git_prefix = ["git"]
if gh_token and repo_url.startswith("https://"):
    b64 = base64.b64encode(f"x-access-token:{gh_token}".encode("utf-8")).decode("ascii")
    git_prefix = ["git", "-c", f"http.extraHeader=Authorization: Basic {b64}"]

if (CODE_DIR / ".git").exists():
    subprocess.run([*git_prefix, "-C", str(CODE_DIR), "remote", "set-url", "origin", repo_url], check=True)
    subprocess.run([*git_prefix, "-C", str(CODE_DIR), "fetch", "--prune", "origin"], check=True)
    rc = subprocess.run([*git_prefix, "-C", str(CODE_DIR), "checkout", git_branch], check=False)
    if rc.returncode != 0:
        subprocess.run([*git_prefix, "-C", str(CODE_DIR), "checkout", "-B", git_branch, f"origin/{git_branch}"], check=True)
    subprocess.run([*git_prefix, "-C", str(CODE_DIR), "pull", "--ff-only", "origin", git_branch], check=True)
else:
    if CODE_DIR.exists():
        shutil.rmtree(CODE_DIR)
    subprocess.run([*git_prefix, "clone", "--branch", git_branch, repo_url, str(CODE_DIR)], check=True)

head_sha = subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
os.chdir(CODE_DIR)
for p in (str(CODE_DIR), str(CODE_DIR / "dcpa-train-py")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ["PYTHONPATH"] = f"{CODE_DIR}:{CODE_DIR / 'dcpa-train-py'}"

# 安装 zstd（加速多核 .tar.zst 打包/解压）与核心 Python 依赖
if shutil.which("zstd") is None:
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "zstd"], check=False)

deps = [
    "safetensors",
    "datasets",
    "huggingface_hub",
    "hf_transfer",
    "zstandard",
    "pyyaml",
    "wandb",
    "pytest",
    "tree-sitter",
    "tree-sitter-python",
    "tree-sitter-javascript",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *deps], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--no-build-isolation", "-e", str(CODE_DIR / "dcpa-train-py")],
    check=True,
)

if run_mode == "export_and_gate6" and shutil.which("cargo") is None:
    print("[Rust] Installing rustup toolchain for Gate 6 cross-stack verification...")
    subprocess.run("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y", shell=True, check=True)
    os.environ["PATH"] = f"{Path.home() / '.cargo' / 'bin'}:{os.environ['PATH']}"

print(
    f"[Deploy] {CODE_DIR} @ {git_branch} ({head_sha}) | "
    f"zstd={'OK' if shutil.which('zstd') else 'py-fallback'} | "
    f"HF_TOKEN={'SET' if os.environ.get('HF_TOKEN') else 'unset'} | "
    f"WANDB_KEY={'SET' if os.environ.get('WANDB_API_KEY') else 'offline/unset'}"
)


In [ ]:
# Cell 3 [阶段 0 CPU 多核造数与打包 OR 阶段 1/2 GPU NVMe 30秒极速恢复]
# 说明：严禁直接 mmap Google Drive FUSE（铁律 #3），所有语料先落本地 NVMe (/content/nvme_cache)！
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

from data.hf_auth import configure_hf_auth
from train.colab_drive_sync import sync_file_to_drive

configure_hf_auth()

RUN_MODE = os.environ.get("DCPA_FLAT_RUN_MODE", "cpu_forge")
DATA_PROFILE = os.environ.get("DCPA_FLAT_DATA_PROFILE", "v3_flat_5g")
ARCHIVE_STORAGE = os.environ.get("DCPA_FLAT_ARCHIVE_STORAGE", "drive")
HF_ARCHIVE_REPO = os.environ.get("DCPA_FLAT_HF_ARCHIVE_REPO", "").strip()
DRIVE_ROOT = Path(os.environ.get("DCPA_FLAT_DRIVE_ROOT", "/content/drive/MyDrive/DCPA_Flat_v3"))
AUTO_UNASSIGN = os.environ.get("DCPA_FLAT_AUTO_UNASSIGN", "1") == "1"

NVME_ROOT = Path("/content/nvme_cache")
NVME_ROOT.mkdir(parents=True, exist_ok=True)
ARCHIVE_NAME = f"dcpa_flat_{DATA_PROFILE}.tar.zst"
LOCAL_ARCHIVE = NVME_ROOT / ARCHIVE_NAME
DRIVE_ARCHIVE = DRIVE_ROOT / "data_archive" / ARCHIVE_NAME

def _stream_run(cmd):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    rc = int(proc.wait())
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

t0 = time.perf_counter()
nvme_ready = (NVME_ROOT / "data" / "train.bin").is_file() and (NVME_ROOT / "data" / "train.loss_mask.bin").is_file()

if RUN_MODE == "cpu_forge":
    # 纯 CPU 多线程抓取最新 2025-2026 语料 + 无损分块 + 合成能力/SFT 炼金分片 + .tar.zst 流式打包
    workers = max(4, min(8, os.cpu_count() or 4))
    # 当存 Drive 时直接流式写盘到 DRIVE_ARCHIVE，不在 /content 落双份大压缩包（为 90G 档位省出 ~38 GB 本地盘）
    if ARCHIVE_STORAGE == "drive":
        DRIVE_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
        pack_target = DRIVE_ARCHIVE
    else:
        pack_target = LOCAL_ARCHIVE
    print(f"[CPU Forge] Building profile={DATA_PROFILE} on {NVME_ROOT} (pack -> {pack_target}, jobs={workers}) ...", flush=True)
    _stream_run([
        sys.executable, "-u", "data/prepare_vast_corpus.py",
        "--profile", DATA_PROFILE,
        "--nvme-root", str(NVME_ROOT),
        "--max-parallel-jobs", str(workers),
        "--pack-archive", str(pack_target),
    ])
    arch_gb = pack_target.stat().st_size / (1024**3)
    print(f"[CPU Forge] Packed {pack_target.name} ({arch_gb:.2f} GB) in {time.perf_counter() - t0:.1f}s.", flush=True)

    if ARCHIVE_STORAGE == "hf_repo":
        from huggingface_hub import HfApi

        api = HfApi()
        api.create_repo(repo_id=HF_ARCHIVE_REPO, repo_type="dataset", private=True, exist_ok=True)
        print(f"[Upload -> HF Private Dataset] Uploading {ARCHIVE_NAME} ({arch_gb:.2f} GB) to {HF_ARCHIVE_REPO} ...", flush=True)
        api.upload_file(
            path_or_fileobj=str(LOCAL_ARCHIVE),
            path_in_repo=ARCHIVE_NAME,
            repo_id=HF_ARCHIVE_REPO,
            repo_type="dataset",
        )
        manifest = NVME_ROOT / "vast_corpus_manifest.json"
        if manifest.is_file():
            api.upload_file(
                path_or_fileobj=str(manifest),
                path_in_repo=f"manifest_{DATA_PROFILE}.json",
                repo_id=HF_ARCHIVE_REPO,
                repo_type="dataset",
            )
        print(f"[Upload DONE] Stored on HF private dataset: {HF_ARCHIVE_REPO}/{ARCHIVE_NAME}", flush=True)
        LOCAL_ARCHIVE.unlink(missing_ok=True)
    else:
        manifest = NVME_ROOT / "vast_corpus_manifest.json"
        if manifest.is_file():
            sync_file_to_drive(manifest, DRIVE_ARCHIVE.parent, dest_name=f"manifest_{DATA_PROFILE}.json")
        print(f"[Upload DONE] Stream-packed directly to Drive: {DRIVE_ARCHIVE} ({arch_gb:.2f} GB)", flush=True)

    if hasattr(os, "sync"):
        os.sync()
    print("\n✅ [阶段 0 CPU 造数圆满完成] 下一步：切换至 RTX Pro 6000 / A100 实例，在 Cell 1 设置 RUN_MODE='killgate_5k'。", flush=True)
    if AUTO_UNASSIGN:
        if ARCHIVE_STORAGE == "drive" and arch_gb >= 10.0:
            print(
                f"[Safety Pause] Drive 大文件 ({arch_gb:.1f} GB) 走 FUSE 后台异步上传，已暂停自动释放 CPU 实例；"
                "请刷新 Google Drive 网页版确认归档大小完整后再手动断开！",
                flush=True,
            )
        else:
            print("[Auto-Unassign] Waiting 10s after os.sync() before releasing CPU runtime...", flush=True)
            time.sleep(10)
            from google.colab import runtime

            runtime.unassign()
else:
    if nvme_ready:
        print(f"[NVMe HIT] Shards already ready at {NVME_ROOT / 'data'} (skipping restore).", flush=True)
    else:
        archive_src: str | None = None
        if ARCHIVE_STORAGE == "hf_repo":
            from huggingface_hub import hf_hub_download

            print(f"[HF Restore] Pulling {ARCHIVE_NAME} from private repo {HF_ARCHIVE_REPO} -> NVMe ...", flush=True)
            archive_src = hf_hub_download(
                repo_id=HF_ARCHIVE_REPO,
                filename=ARCHIVE_NAME,
                repo_type="dataset",
                local_dir=str(NVME_ROOT / "_dl"),
            )
        elif DRIVE_ARCHIVE.is_file():
            # 直接从 Drive 流式解压到本地 NVMe（零双倍本地盘占用，为 90G 档位省出 ~38 GB /content 空间）
            print(f"[Drive Restore] Streaming extract {DRIVE_ARCHIVE} -> {NVME_ROOT} (zero extra local copy) ...", flush=True)
            archive_src = str(DRIVE_ARCHIVE)

        if archive_src:
            _stream_run([
                sys.executable, "-u", "data/prepare_vast_corpus.py",
                "--profile", DATA_PROFILE,
                "--nvme-root", str(NVME_ROOT),
                "--from-archive", archive_src,
            ])
            LOCAL_ARCHIVE.unlink(missing_ok=True)
            shutil.rmtree(NVME_ROOT / "_dl", ignore_errors=True)
            print(f"[Restore DONE] NVMe shards verified in {time.perf_counter() - t0:.1f}s.", flush=True)
        else:
            raise FileNotFoundError(
                f"Archive {ARCHIVE_NAME} not found in {ARCHIVE_STORAGE}! "
                "请先用无 GPU CPU 实例把 RUN_MODE 设为 'cpu_forge' 生成归档，避免在昂贵 GPU 上空等下载。"
            )


In [ ]:
# Cell 4 [60 秒上线前复检：DCPA-Flat v3.3 单元测试 + 4 步端到端冒烟]
import os
import shutil
import subprocess
import sys
import time

RUN_MODE = os.environ.get("DCPA_FLAT_RUN_MODE", "cpu_forge")
if RUN_MODE == "cpu_forge":
    print("[Skip] RUN_MODE='cpu_forge' 无需运行 GPU 训练前复检。")
else:
    t0 = time.perf_counter()
    tests = [
        "tests/test_flat_v3_interfaces.py",
        "tests/test_flat_ws1_data.py",
        "tests/test_flat_ws2_grid_and_ring.py",
        "tests/test_flat_ws3_carrier_and_stack.py",
        "tests/test_flat_ws4_hippocampus.py",
        "tests/test_flat_ws5_integration.py",
    ]
    subprocess.run([sys.executable, "-u", "-m", "pytest", "-q", *tests], check=True)
    smoke_dir = "/tmp/dcpa_flat_preflight_smoke"
    shutil.rmtree(smoke_dir, ignore_errors=True)
    subprocess.run(
        [sys.executable, "-u", "train/train_flat.py", "--smoke", "--local-cache-dir", smoke_dir],
        check=True,
    )
    shutil.rmtree(smoke_dir, ignore_errors=True)
    print(f"[Preflight GREEN] DCPA-Flat v3.3 WS-1..WS-5 + 4-step Trainer Smoke 100% PASSED in {time.perf_counter() - t0:.1f}s.")


In [ ]:
# Cell 5 [核心 GPU 执行：5k 三路生死门 (killgate_5k) OR 48M / 206M 五阶段旗舰训练 + 完工自动释放显卡]
import json
import os
import shlex
import subprocess
import sys
import time
from pathlib import Path

from train.colab_drive_sync import sync_file_to_drive

RUN_MODE = os.environ.get("DCPA_FLAT_RUN_MODE", "cpu_forge")
DRIVE_ROOT = Path(os.environ.get("DCPA_FLAT_DRIVE_ROOT", "/content/drive/MyDrive/DCPA_Flat_v3"))
NVME_ROOT = Path("/content/nvme_cache")
AUTO_UNASSIGN = os.environ.get("DCPA_FLAT_AUTO_UNASSIGN", "1") == "1"

def _stream_run(cmd) -> int:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    return int(proc.wait())

exit_code = -1
try:
    if RUN_MODE in ("cpu_forge", "export_and_gate6"):
        print(f"[Skip] RUN_MODE={RUN_MODE!r} 不执行 Cell 5 训练主循环。")
        exit_code = 0
    elif RUN_MODE == "killgate_5k":
        report_local = NVME_ROOT / "killgate_5k_report.json"
        cmd = [
            sys.executable, "-u", "gates/run_flat_killgate_5k.py",
            "--bin", str(NVME_ROOT / "data/train.bin"),
            "--idx", str(NVME_ROOT / "data/train.idx"),
            "--val-bin", str(NVME_ROOT / "data/val.bin"),
            "--val-idx", str(NVME_ROOT / "data/val.idx"),
            "--steps", "5000",
            "--warmup-steps", "800",
            "--plugin-steps", "200",
            "--out-json", str(report_local),
        ]
        print(f"[Killgate 5k] Launching: {' '.join(cmd)}", flush=True)
        exit_code = _stream_run(cmd)
        if report_local.is_file():
            sync_file_to_drive(report_local, DRIVE_ROOT / "logs/killgate_5k", dest_name="killgate_5k_report.json")
            rep = json.loads(report_local.read_text(encoding="utf-8"))
            rc = rep["runs"]["run_c"]
            rb = rep["runs"]["run_b"]
            ra = rep["runs"]["run_a"]
            gb = rep["gate_b_stress"]
            ge = rep["gate_e_plugin"]
            cmp_ = rep["comparisons"]
            print("\n================ DCPA-Flat v3.3 5k 生死门裁决单 ================")
            print(f"Gate A 吞吐/溢出 : Run C steady={rc['steady_kb_s']:.1f} KB/s (>=360) | ovf={rc['overflow_byte_ratio']:.5f} (<0.001)")
            print(f"Gate B 512步梯度 : passed={gb['passed']} | max_grad={gb.get('max_grad_norm', 0):.2f} | ret64={gb.get('grad_retention_64', 0):.3f} (>=0.05)")
            print(f"Gate C 槽位/均衡 : empty_write={rc['empty_write_rate']:.3f} (>=0.85) | expert_cv={rc['expert_cv']:.3f} (<=0.15)")
            print(f"Gate D 三路对比  : C vs A EN/ZH Δ={cmp_['run_c_vs_a_en_zh_bpc_delta']:+.3f} (>=+0.20) | Code Δ={cmp_['run_c_vs_a_code_bpc_delta']:+.3f} (>=+0.15)")
            print(f"Gate D 窗口判定  : C vs B EN/ZH Δ={cmp_['run_c_vs_b_en_zh_bpc_delta']:+.3f} | Code Δ={cmp_['run_c_vs_b_code_bpc_delta']:+.3f}")
            print(f"Gate E 插件零漂移: target BPC {ge['target_bpc_before']:.3f} -> {ge['target_bpc_after']:.3f} | anchor_drift={ge['anchor_max_logit_drift']:.3e} (==0.0)")
            print("=================================================================")
        if exit_code != 0:
            raise RuntimeError(f"Killgate 5k exited with code {exit_code}")
    else:
        tag = "flat_48m" if RUN_MODE == "train_48m" else "flat_206m"
        cfg_file = "configs/v3/dcpa_flat_48m_v3.yaml" if RUN_MODE == "train_48m" else "configs/v3/dcpa_flat_206m_v3.yaml"
        cmd = [
            sys.executable, "-u", "train/train_flat.py",
            "--config", cfg_file,
            "--data-dir", str(NVME_ROOT / "data"),
            "--cap-data-dir", str(NVME_ROOT / "cap_data"),
            "--sft-data-dir", str(NVME_ROOT / "sft_data"),
            "--local-cache-dir", str(NVME_ROOT / f"run_{tag}"),
            "--drive-sync-dir", str(DRIVE_ROOT / "checkpoints" / tag),
            "--drive-logs-dir", str(DRIVE_ROOT / "logs" / tag),
            "--export-dir", str(NVME_ROOT / "export" / tag),
            "--drive-export-dir", str(DRIVE_ROOT / "export" / tag),
            "--resume", "auto",
        ]
        if os.environ.get("DCPA_FLAT_USE_WANDB", "1") == "1":
            cmd.extend(["--wandb", "--wandb-project", os.environ.get("WANDB_PROJECT", "dcpa-flat-v3")])
            if os.environ.get("WANDB_NAME"):
                cmd.extend(["--wandb-run-name", os.environ["WANDB_NAME"]])
        else:
            cmd.append("--no-wandb")
        if os.environ.get("DCPA_FLAT_LOCAL_WINDOW") in ("true", "false"):
            cmd.extend(["--use-local-window-layers", os.environ["DCPA_FLAT_LOCAL_WINDOW"]])
        if os.environ.get("DCPA_FLAT_GRAD_GUARD") in ("none", "router_sg", "soft_clip"):
            cmd.extend(["--carrier-grad-guard", os.environ["DCPA_FLAT_GRAD_GUARD"]])
        if os.environ.get("DCPA_FLAT_TOTAL_STEPS"):
            cmd.extend(["--total-steps", os.environ["DCPA_FLAT_TOTAL_STEPS"]])
        extra = os.environ.get("DCPA_FLAT_EXTRA_ARGS", "").strip()
        if extra:
            cmd.extend(shlex.split(extra))
        print(f"[Train {tag}] Launching: {' '.join(cmd)}", flush=True)
        exit_code = _stream_run(cmd)
        if exit_code != 0:
            raise RuntimeError(f"train_flat.py exited with code {exit_code}")
        print(f"[Train {tag}] Completed cleanly! Rust-ready export mirrored to {DRIVE_ROOT / 'export' / tag}", flush=True)
finally:
    if hasattr(os, "sync"):
        os.sync()
    if AUTO_UNASSIGN and exit_code == 0 and RUN_MODE in ("killgate_5k", "train_48m", "train_206m"):
        print("[Auto-Unassign] Waiting 10s after os.sync() before releasing GPU to save Colab credits...", flush=True)
        time.sleep(10)
        from google.colab import runtime

        runtime.unassign()


In [ ]:
# Cell 6 [可选：Gate 6 跨栈数值对齐验证 + 领域插件热插拔微调]
import os
import subprocess
import sys
from pathlib import Path

RUN_MODE = os.environ.get("DCPA_FLAT_RUN_MODE", "cpu_forge")
DRIVE_ROOT = Path(os.environ.get("DCPA_FLAT_DRIVE_ROOT", "/content/drive/MyDrive/DCPA_Flat_v3"))
NVME_ROOT = Path("/content/nvme_cache")

if RUN_MODE != "export_and_gate6":
    print(f"[Skip] 设置 RUN_MODE='export_and_gate6' 后再运行本 Cell。")
else:
    # 1. 运行 Python <-> Rust (no_std) Gate 6 全量对齐 + 0 堆分配 + 睡眠守护闭环验收
    subprocess.run([sys.executable, "-u", "gates/gate6_flat_v3_alignment.py"], check=True)

    # 2. 若 Drive 已有导出的 206M 或 48M 底座，示范训练一个 SQL/代码领域外挂插件 (.dcpa_expert.safetensors)
    for tag in ("flat_206m", "flat_48m"):
        base_dir = DRIVE_ROOT / "export" / tag
        cap_bin = NVME_ROOT / "cap_data/cap_code_train.bin"
        cap_idx = NVME_ROOT / "cap_data/cap_code_train.idx"
        if (base_dir / "model.safetensors").is_file() and cap_bin.is_file() and cap_idx.is_file():
            out_plugin = base_dir / "code_domain.dcpa_expert.safetensors"
            subprocess.run([
                sys.executable, "-u", "-m", "dcpa.train_plugin",
                "--mode", "domain",
                "--base-dir", str(base_dir),
                "--replay-bin", str(cap_bin),
                "--replay-idx", str(cap_idx),
                "--out-plugin", str(out_plugin),
                "--pack-id", "code_domain_v1",
                "--steps", "100",
                "--device", "auto",
            ], check=True)
            print(f"[Plugin DONE] Saved zero-drift domain plugin -> {out_plugin}")
            break
